# 01 — Explore BC5CDR

Token-level look at the BC5CDR NER data: example sentences, tag distribution,
sentence lengths, and entity counts per split.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AleyGabr04/med-nlp-01-clinical-ner.git"

if IN_COLAB:
    if not os.path.exists("med-nlp-01-clinical-ner"):
        !git clone -q {REPO_URL}
    %cd med-nlp-01-clinical-ner
    %pip install -q .
else:
    %cd ..

In [ ]:
import matplotlib.pyplot as plt

from medtag.config import load_config
from medtag.data import (
    entity_counts,
    get_label_names,
    label_counts,
    load_bc5cdr,
    sentence_lengths,
    split_summary,
)

data_cfg = load_config("configs/config.yaml")["data"]
label_names = get_label_names(data_cfg)
dsd = load_bc5cdr(data_cfg)
dsd

## Example sentences

In [ ]:
def show_sentence(example: dict) -> None:
    for token, tag_id in zip(example["tokens"], example["tags"]):
        tag = label_names[tag_id]
        marker = "  <--" if tag != "O" else ""
        print(f"{token:<25} {tag}{marker}")
    print()


for i in [0, 1, 7]:
    show_sentence(dsd["train"][i])

## Split summary

In [ ]:
split_summary(dsd, label_names)

## Tag distribution (train)

In [ ]:
counts = label_counts(dsd["train"], label_names)
print(counts)
print(f"\nNon-O share of tokens: {1 - counts['O'] / counts.sum():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
counts.plot.bar(ax=axes[0], title="All tags")
counts.drop("O").plot.bar(ax=axes[1], title="Entity tags only")
for ax in axes:
    ax.set_ylabel("tokens")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()

## Sentence lengths

In [ ]:
lengths = {split: sentence_lengths(ds) for split, ds in dsd.items()}
print(lengths["train"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lengths["train"], bins=60)
ax.set_xlabel("tokens per sentence")
ax.set_ylabel("sentences")
ax.set_title("Sentence length (train, word tokens)")
plt.tight_layout()

## Entity counts per type

In [ ]:
import pandas as pd

entities = pd.DataFrame({split: entity_counts(ds, label_names) for split, ds in dsd.items()})
print(entities)

entities.T.plot.bar(figsize=(7, 3.5), title="Entity mentions per split")
plt.ylabel("entities")
plt.xticks(rotation=0)
plt.tight_layout()